# NB00 · Setup sample data
Companion to **Fabric Spark on One Page**.

Creates a small sample lakehouse (2,000 orders, 200 customers) so every code card in the book can run. The other notebooks call this one with `%run`, so you rarely run it on its own.

**Before you start:** attach a *test* lakehouse as the default lakehouse. This notebook overwrites these tables in it: `sales`, `silver_sales`, `orders`, `customers`, `crm_customers`, `silver_customers`, `dim_customer`, `dim_country`, `bronze_orders`, `silver_orders`, `gold_revenue`, `gold_sales_history`; and the folders `Files/raw`, `Files/landing`, `Files/archive`, `Files/export`, `Files/tmp`.

In [ ]:
import json
# Builds a small, repeatable sample lakehouse for testing every code card.
# Safe to re-run: it overwrites only the sample tables and files listed here.
import datetime
import random

from delta.tables import DeltaTable
from pyspark.sql import Window as W
from pyspark.sql import functions as F
from pyspark.sql import types as T

random.seed(42)
COUNTRIES = {"UK": "United Kingdom", "IE": "Ireland", "US": "United States", "DE": "Germany", "IN": "India"}
REGION = {"UK": "EU", "IE": "EU", "DE": "EU", "US": "NA", "IN": "APAC"}
PRODUCTS = [("SKU-001", "Laptop"), ("SKU-002", "Monitor"), ("SKU-003", "Keyboard"),
            ("SKU-004", "Mouse"), ("SKU-005", "Dock")]
TODAY = datetime.date(2026, 10, 19)

SALES_SCHEMA = T.StructType([
    T.StructField("order_id", T.StringType()),
    T.StructField("order_date", T.DateType()),
    T.StructField("amount", T.DoubleType()),
    T.StructField("customer_id", T.StringType()),
    T.StructField("country", T.StringType()),
    T.StructField("region", T.StringType()),
    T.StructField("product", T.StringType()),
    T.StructField("sku", T.StringType()),
    T.StructField("qty", T.IntegerType()),
    T.StructField("year", T.IntegerType()),
    T.StructField("updated_at", T.TimestampType()),
])


def _make_rows(n=2000):
    rows, start = [], datetime.date(2025, 1, 1)
    for i in range(1, n + 1):
        cc = random.choice(list(COUNTRIES))
        sku, product = random.choice(PRODUCTS)
        d = TODAY if i > n - 10 else start + datetime.timedelta(days=random.randint(0, 655))
        r = random.random()  # skew: C001 has ~20% of orders, ~2% have no customer
        cust = None if r < 0.02 else ("C001" if r < 0.22 else f"C{random.randint(2, 200):03d}")
        ts = datetime.datetime.combine(d, datetime.time(random.randint(0, 23), random.randint(0, 59)))
        rows.append((f"O{i:05d}", d, round(random.uniform(5, 2500), 2), cust, cc, REGION[cc],
                     product, sku, random.randint(1, 10), d.year, ts))
    return rows


_ROWS = _make_rows()


def sample_sales():
    """2,000 orders, Jan 2025 to 19 Oct 2026, with a skewed key and some null keys."""
    return spark.createDataFrame(_ROWS, SALES_SCHEMA)


def _csv(rows, cols):
    def v(x):
        return "" if x is None else str(x)
    return "\n".join([",".join(cols)] + [",".join(v(x) for x in r) for r in rows]) + "\n"


def _save(df, name):
    df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(name)


fs = notebookutils.fs

# ---- Files/
for folder in ["Files/raw", "Files/landing", "Files/archive", "Files/export", "Files/tmp"]:
    try:
        fs.rm(folder, True)
    except Exception:
        pass

cols = ["order_id", "order_date", "amount", "customer_id", "country", "region", "product", "sku", "qty"]
fs.put("Files/raw/sales.csv", _csv([r[:9] for r in _ROWS], cols), True)
fs.put("Files/raw/orders.csv", _csv([r[:9] for r in _ROWS], cols), True)
fs.put("Files/raw/a.csv", "id,value\n1,a\n", True)
fs.put("Files/raw/b.csv", "id,value\n2,b\n", True)
fs.put("Files/tmp/scratch.txt", "temporary file\n", True)
fs.put("Files/raw/events.json", json.dumps(
    [{"event_id": i, "type": random.choice(["view", "click", "buy"]), "ts": f"2026-10-19T10:{i:02d}:00"}
     for i in range(20)], indent=2), True)
sample_sales().write.mode("overwrite").parquet("Files/raw/sales/")

land_cols = ["order_id", "order_date", "order_month", "customer_id", "country", "region",
             "product", "amount", "updated_at"]
land = [(r[0], r[1], r[1].strftime("%Y-%m"), r[3], r[4], r[5], r[6], r[2],
         r[10].strftime("%Y-%m-%d %H:%M:%S")) for r in _ROWS]
fs.put("Files/landing/orders/orders_1.csv", _csv(land[:1200], land_cols), True)
fs.put("Files/landing/orders/orders_2.csv", _csv(land[1100:], land_cols), True)  # 100 duplicate ids

# ---- Tables/
sales = sample_sales()
_save(sales, "sales")
_save(sales, "silver_sales")

orders = (sales.withColumn("country_code", F.col("country"))
               .withColumn("cust_id", F.regexp_extract("customer_id", r"(\d+)", 1).cast("int")))
_save(orders, "orders")

cust_rows = [(f"C{i:03d}", i, f"customer {i}", random.choice(list(REGION.values())),
              random.choice(["retail", "trade", "online"]), random.choice(["bronze", "silver", "gold"]))
             for i in range(1, 201)]
customers_df = spark.createDataFrame(cust_rows, "customer_id string, id int, name string, region string, segment string, tier string")
_save(customers_df, "customers")
_save(customers_df, "crm_customers")          # stands in for a shortcut on page 6
_save(customers_df.select("customer_id", "name", "region", "tier"), "silver_customers")
_save(customers_df.select("customer_id", "segment", F.lit(True).alias("is_current"))
      .unionByName(customers_df.limit(20).select("customer_id", F.lit("old").alias("segment"),
                                                 F.lit(False).alias("is_current"))), "dim_customer")
_save(spark.createDataFrame(list(COUNTRIES.items()), "country_code string, country_name string"), "dim_country")

bronze = (spark.read.option("header", True).csv("Files/landing/orders/")
          .withColumn("_ingested_at", F.current_timestamp())
          .withColumn("_source_file", F.col("_metadata.file_path")))
_save(bronze, "bronze_orders")
_w = W.partitionBy("order_id").orderBy(F.desc("_ingested_at"))
_save(spark.read.table("bronze_orders")
      .withColumn("amount", F.col("amount").cast("decimal(18,2)"))
      .withColumn("rn", F.row_number().over(_w)).filter("rn = 1").drop("rn"), "silver_orders")
_save(sales.groupBy("country", "year").agg(F.sum("amount").alias("revenue")), "gold_revenue")
(sales.write.mode("overwrite").option("overwriteSchema", "true")
      .partitionBy("year").saveAsTable("gold_sales_history"))
spark.sql("DROP TABLE IF EXISTS stg_enriched")

# ---- DataFrames and views the cards use by name
orders = spark.read.table("orders")
customers = spark.read.table("customers")
dim_country = spark.read.table("dim_country")
df = sample_sales()

print("Sample lakehouse ready:",
      ", ".join(t.name for t in spark.catalog.listTables()))